In [2]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [3]:
from pathlib import Path
import zipfile

drive_root = Path("/content/gdrive/MyDrive")

mission_zips = list(drive_root.rglob("Mission1.zip"))
print("찾은 Mission1.zip:", mission_zips)

mission_zip = mission_zips[0]
extract_dir = Path("/content/Mission1")

if not extract_dir.exists():
    with zipfile.ZipFile(mission_zip, "r") as z:
        z.extractall(extract_dir)

print("압축 해제 완료")

찾은 Mission1.zip: [PosixPath('/content/gdrive/MyDrive/Mission1.zip')]
압축 해제 완료


In [7]:
extract_dir = Path("/content/대학부_데이터_통합")
extract_dir.mkdir(parents=True, exist_ok=True)

for i, zip_path in enumerate(data_zips, 1):
    print(f"{i}/{len(data_zips)} 압축 해제 중: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

print("압축 해제 완료")

1/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-001.zip
2/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-002.zip
3/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-003.zip
4/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-004.zip
5/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-005.zip
6/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-006.zip
7/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-007.zip
8/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-008.zip
9/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-009.zip
10/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-010.zip
11/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-011.zip
12/12 압축 해제 중: 대학부 데이터-20261010T074933Z-1-012.zip
압축 해제 완료


In [14]:
import subprocess
import sys

output_path = "/content/gdrive/MyDrive/mission1_validation_full_result.csv"

command = [
    sys.executable,
    str(inference_py),
    "--wav_dir", str(valid_wav),
    "--json_dir", str(valid_json),
    "--model_dir", str(model_dir),
    "--output", output_path,
]

print("전체 Validation 추론 시작")
result = subprocess.run(command, text=True)

print("종료 코드:", result.returncode)
print("결과 파일:", output_path)

전체 Validation 추론 시작
종료 코드: 0
결과 파일: /content/gdrive/MyDrive/mission1_validation_full_result.csv


In [20]:
import pandas as pd
import json

result_path = "/content/gdrive/MyDrive/mission1_validation_full_result.csv"

result_df = pd.read_csv(result_path)

truth = {}

for json_path in valid_json.rglob("*.json"):
    with json_path.open("r", encoding="utf-8-sig") as f:
        item = json.load(f)

    sample_id = str(item.get("_id", ""))
    gender = str(item.get("gender", "")).upper()

    if gender in ["M", "F"]:
        truth[sample_id] = gender

result_df["true_gender"] = result_df["id"].astype(str).map(truth)

matched = result_df.dropna(subset=["true_gender"])

accuracy = (
    matched["gender"] == matched["true_gender"]
).mean()

print("예측 개수:", len(result_df))
print("정답 연결 개수:", len(matched))
print("전체 Validation Accuracy:", accuracy)
print("정확도(%):", accuracy * 100)

display(matched.head(20))

예측 개수: 3640
정답 연결 개수: 3640
전체 Validation Accuracy: 0.9763736263736263
정확도(%): 97.63736263736263


,id,gender,true_gender
0,651e464d69a4f266f0626837,M,M
1,651e464d69a4f266f0626845,M,M
2,651e464d69a4f266f0626846,F,F
3,651e464d69a4f266f0626859,F,F
4,651e464d69a4f266f062685e,M,M
5,651e464d69a4f266f062685f,F,F
6,651e464d69a4f266f062686f,M,M
7,651e464d69a4f266f0626889,M,M
8,651e464d69a4f266f06268ba,F,F
9,651e464d69a4f266f06268be,F,F
